# ✈️ AeroMech — Cessna 172 Agentic RAG (Colab, free)

Ground-based maintenance & training assistant. Free tools only:
**Google Gemini** (free LLM) · **BGE embeddings** · **NumPy vector search** ·
**BM25** · **cross-encoder rerank** · **LangGraph** · **Streamlit**.

> ⚠️ Training/maintenance study tool only. Not for in-flight use.

### Setup (one time)
1. Get a free Gemini key: https://aistudio.google.com/app/apikey → **Create API key** → copy it.
2. In Colab, click the **🔑 key icon** (left sidebar) → **+ Add new secret**.
3. Name it exactly `GOOGLE_API_KEY`, paste your key, toggle **Notebook access** on.
4. Then run every cell top to bottom: **Runtime → Run all**.

This notebook uses **NumPy** for vector search instead of a vector database —
fewer moving parts, nothing to break between library versions.

## 1. Install dependencies

In [4]:
!pip install -q \
  langgraph langchain langchain-google-genai \
  langchain-text-splitters \
  sentence-transformers rank-bm25 numpy streamlit
print("\n Installed. (Ignore any yellow dependency warnings.)")


 Installed. (Ignore any yellow dependency warnings.)


## 2. Load your Gemini API key

In [6]:
import os
from google.colab import userdata
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("GOOGLE_API_KEY loaded.")
except Exception as e:
    print("x Add GOOGLE_API_KEY in the Secrets panel (key icon, left).", e)

GOOGLE_API_KEY loaded.


## 3. Knowledge base
A synthetic sample is written below so it runs out of the box. To use real data,
upload Cessna 172 POH **PDFs** to `/content/data/raw/` (Files panel) and re-run
from here — the next cell reads PDFs too.

In [7]:
import pathlib
pathlib.Path("/content/data/raw").mkdir(parents=True, exist_ok=True)

SAMPLE = """# Cessna 172 Skyhawk Knowledge Base (SYNTHETIC SAMPLE)

> Synthetic sample for a software demo. NOT an official POH.

## Engine Overview
The Cessna 172S is powered by a Lycoming IO-360-L2A engine producing 180
horsepower at 2700 RPM. Normal oil pressure in cruise is 50 to 90 PSI. Oil
temperature should stay below 245 degrees Fahrenheit.

## Engine Failure Training Reference (ABCDE)
The memorized flow taught for engine failure is ABCDE: Airspeed (best glide
about 68 knots), Best field, Checklist, Declare (squawk 7700 and broadcast on
121.5), Execute. This is a TRAINING reference only; in a real emergency pilots
rely on memorized procedures, not external devices.

## Rough Engine / Partial Power Loss
Causes include fouled spark plugs, a stuck valve, or fuel contamination. Ground
check: a normal magneto drop is 125 RPM maximum, with no more than 50 RPM
differential between the left and right magnetos.

## Low Oil Pressure
A reading below 20 PSI indicates a serious problem; do not operate the engine.
Causes include a failed oil pump, clogged filter, or low oil. Minimum oil for
flight is 5 quarts; capacity is 8 quarts.

## Fuel System
The aircraft holds 56 gallons total, 53 usable, across two wing tanks. The fuel
selector positions are LEFT, RIGHT, and BOTH; normal setting is BOTH. Fuel grade
is 100LL aviation gasoline.

## Magneto / Ignition
Dual magnetos provide redundancy. During runup at 1800 RPM a dead magneto (zero
RPM drop) is a no-go item and requires maintenance.
"""
pathlib.Path("/content/data/raw/cessna172_sample.md").write_text(SAMPLE)
print(" Sample knowledge base written.")

 Sample knowledge base written.


## 4. Config

In [8]:
LLM_MODEL    = "gemini-2.5-flash"            # free tier
EMBED_MODEL  = "BAAI/bge-small-en-v1.5"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHUNK_SIZE, CHUNK_OVERLAP = 800, 120
TOP_K_VECTOR, TOP_K_BM25, TOP_K_FINAL = 8, 8, 4

INFLIGHT_REFUSAL = (
    " This assistant is for GROUND-BASED maintenance and training only. "
    "If you are experiencing an in-flight emergency, do NOT consult this tool. "
    "Fly the aircraft, run your memorized checklist, contact ATC. "
    "Aviate, Navigate, Communicate."
)

## 5. Load & chunk documents

In [9]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.6 MB/s eta 0:00:00


In [10]:
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

def load_and_chunk():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "])
    files = (glob.glob("/content/data/raw/*.pdf") +
             glob.glob("/content/data/raw/*.md") +
             glob.glob("/content/data/raw/*.txt"))
    chunks = []
    for f in files:
        name = f.split("/")[-1]
        if f.endswith(".pdf"):
            pages = [(p.extract_text() or "", i+1) for i, p in enumerate(PdfReader(f).pages)]
        else:
            pages = [(open(f, encoding="utf-8").read(), None)]
        for text, page in pages:
            for j, piece in enumerate(splitter.split_text(text)):
                if piece.strip():
                    cid = f"{name}_p{page}_c{j}" if page else f"{name}_c{j}"
                    chunks.append({"id": cid, "text": piece.strip(),
                                   "source": name, "page": page or -1})
    return chunks

chunks = load_and_chunk()
doc_texts = [c["text"] for c in chunks]
print(f" {len(chunks)} chunks from {len(set(c['source'] for c in chunks))} file(s)")

 3 chunks from 1 file(s)


## 6. Build indexes (embeddings + BM25)
First run downloads the embedding model (~1 min). Vectors are stored in a plain
NumPy array — no vector database, nothing to misconfigure.

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

print("Loading embedding model (first time downloads it)...")
embedder = SentenceTransformer(EMBED_MODEL)

doc_vecs = embedder.encode(doc_texts, normalize_embeddings=True,
                           show_progress_bar=True)
doc_vecs = np.asarray(doc_vecs, dtype=np.float32)

def tokenize(t): return [x for x in t.lower().split() if x]
bm25 = BM25Okapi([tokenize(t) for t in doc_texts])

print(f" Indexed {len(doc_texts)} chunks. Vector matrix: {doc_vecs.shape}")

Loading embedding model (first time downloads it)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 Indexed 3 chunks. Vector matrix: (3, 384)


## 7. Hybrid retriever (vector + BM25 → RRF → rerank)

In [12]:
from sentence_transformers import CrossEncoder
print("Loading reranker (first time downloads it)...")
reranker = CrossEncoder(RERANK_MODEL)

def rrf(rank_lists, k=60):
    s = {}
    for ranks in rank_lists:
        for r, idx in enumerate(ranks):
            s[idx] = s.get(idx, 0.0) + 1.0 / (k + r + 1)
    return s

def retrieve(query, top_k=TOP_K_FINAL):
    # dense: cosine via normalized dot product
    qv = embedder.encode([query], normalize_embeddings=True)
    qv = np.asarray(qv, dtype=np.float32)[0]
    sims = doc_vecs @ qv
    vec_rank = list(np.argsort(-sims)[:TOP_K_VECTOR])

    # sparse: BM25
    bscores = bm25.get_scores(tokenize(query))
    bm_rank = list(np.argsort(-bscores)[:TOP_K_BM25])

    # fuse + rerank
    fused = rrf([vec_rank, bm_rank])
    cand = sorted(fused, key=lambda x: fused[x], reverse=True)
    pairs = [(query, doc_texts[i]) for i in cand]
    rr = reranker.predict(pairs)
    order = list(np.argsort(-rr)[:top_k])
    return [{"text": doc_texts[cand[i]], "source": chunks[cand[i]]["source"],
             "page": chunks[cand[i]]["page"], "score": float(rr[i])} for i in order]

print("\nTest retrieval:")
for r in retrieve("normal oil pressure"):
    print(f"  [{r['score']:.2f}] {r['text'][:70]}...")

Loading reranker (first time downloads it)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Test retrieval:
  [3.40] # Cessna 172 Skyhawk Knowledge Base (SYNTHETIC SAMPLE)

> Synthetic sa...
  [2.12] ## Rough Engine / Partial Power Loss
Causes include fouled spark plugs...
  [-11.33] ## Magneto / Ignition
Dual magnetos provide redundancy. During runup a...


## 8. Agentic graph (safety gate → retrieve → generate)

In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI

INFLIGHT_SIGNALS = ["right now","currently","in flight","in the air","engine just",
    "losing altitude","going down","mid air","midair","airborne and","happening now"]
def inflight(text):
    t = text.lower(); return any(s in t for s in INFLIGHT_SIGNALS)

def kb_search(query):
    res = retrieve(query)
    if not res: return "No relevant passages found."
    return "\n\n".join(
        f"[{i}] ({r['source']}" + (f", p.{r['page']}" if r['page'] and r['page']>0 else "") +
        f")\n{r['text']}" for i, r in enumerate(res, 1))

SYSTEM = ("You are AeroMech, a ground-based maintenance and training assistant for "
    "the Cessna 172. Answer ONLY from the CONTEXT. If the answer is not there, say "
    "you don't have that information and suggest the official POH. Cite bracketed "
    "sources like [1]. Never give in-flight emergency advice. Be concise and use "
    "exact figures from the context.")

def llm():
    return ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0.1)

class S(TypedDict):
    question: str; context: str; answer: str; route: str

def gate(s):    return {**s, "route": "refuse" if inflight(s["question"]) else "retrieve"}
def refuse(s):  return {**s, "answer": INFLIGHT_REFUSAL}
def retr(s):    return {**s, "context": kb_search(s["question"])}
def gen(s):
    p = f"{SYSTEM}\n\nCONTEXT:\n{s['context']}\n\nQUESTION: {s['question']}\n\nANSWER:"
    return {**s, "answer": llm().invoke(p).content}

g = StateGraph(S)
for n, fn in [("gate",gate),("refuse",refuse),("retrieve",retr),("generate",gen)]:
    g.add_node(n, fn)
g.set_entry_point("gate")
g.add_conditional_edges("gate", lambda s: s["route"], {"refuse":"refuse","retrieve":"retrieve"})
g.add_edge("refuse", END); g.add_edge("retrieve","generate"); g.add_edge("generate", END)
agent = g.compile()

def ask(q):
    return agent.invoke({"question": q, "context": "", "answer": "", "route": ""})
print(" Agent ready.")

 Agent ready.


## 9. Try it

In [14]:
print("Q: What is normal oil pressure in cruise?")
print(ask("What is normal oil pressure in cruise?")["answer"])

print("\n" + "="*60)
print("Q (fake in-flight emergency — should be refused):")
print(ask("my engine just failed in flight right now, what do I do?")["answer"])

Q: What is normal oil pressure in cruise?
Normal oil pressure in cruise is 50 to 90 PSI [1].

Q (fake in-flight emergency — should be refused):
 This assistant is for GROUND-BASED maintenance and training only. If you are experiencing an in-flight emergency, do NOT consult this tool. Fly the aircraft, run your memorized checklist, contact ATC. Aviate, Navigate, Communicate.


## 10. Evaluate retrieval quality
Hit-rate and MRR against a small gold set — no LLM needed, so it's fast and free.
Putting these numbers in your README is a strong signal to reviewers.

In [15]:
testset = [
  {"q":"What is normal oil pressure in cruise?","gt":"50 to 90 PSI"},
  {"q":"What engine powers the 172S and its horsepower?","gt":"Lycoming IO-360 180 horsepower"},
  {"q":"What is the maximum normal magneto drop?","gt":"125 RPM 50 differential"},
  {"q":"How much usable fuel and what grade?","gt":"53 gallons 100LL"},
  {"q":"Minimum oil quantity for flight?","gt":"5 quarts 8 capacity"},
  {"q":"What does ABCDE stand for?","gt":"Airspeed Best Checklist Declare Execute"},
]
def metrics(samples):
    hits, rr = 0, 0.0
    for s in samples:
        gt = set(s["gt"].lower().split())
        for rank, r in enumerate(retrieve(s["q"]), 1):
            if len(gt & set(r["text"].lower().split())) >= max(2, len(gt)//3):
                hits += 1; rr += 1.0/rank; break
    n = len(samples)
    return {"hit_rate": round(hits/n, 3), "mrr": round(rr/n, 3), "n": n}

print("Retrieval metrics:", metrics(testset))

Retrieval metrics: {'hit_rate': 1.0, 'mrr': 1.0, 'n': 6}


---
### Where to go next
- **Use real data:** upload POH PDFs to `/content/data/raw/`, re-run from cell 3.
- **Permanent demo:** push this to GitHub, deploy `app.py` (a Streamlit version)
  to **Streamlit Community Cloud** (free) with your Gemini key as a secret.
- **Note for reviewers:** copy the retrieval metrics above into your README.

Colab wipes its VM on disconnect — just **Runtime → Run all** to rebuild.
Gemini's free tier allows ~10–15 requests/min; if a burst returns a 429, wait a
minute and re-run that cell.